# Colab setup — Waste Classification

Chạy notebook này **một lần ở đầu mỗi session Colab**. Notebook sẽ:

1. Mount Google Drive.
2. Clone hoặc pull repo từ GitHub.
3. Cài `uv` và các thư viện trong `requirements-colab.txt` (giữ torch có sẵn của Colab).
4. Copy bundle dữ liệu từ Drive vào `/content/data` và giải nén.
5. Kiểm tra GPU và in metadata môi trường.

**Chuẩn bị trước:** upload `dist/waste_raw_v1.zip` lên
`MyDrive/waste_classifier/data/waste_raw_v1.zip`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/QuangMinh03042005/waste_classifier.git"
PROJECT_DIR = Path("/content/waste_classifier")
DRIVE_ROOT = Path("/content/drive/MyDrive/waste_classifier")
BUNDLE = DRIVE_ROOT / "data" / "waste_raw_v1.zip"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Repo:", PROJECT_DIR)

In [ ]:
!pip install -q uv
!uv pip install --system -r requirements-colab.txt

In [ ]:
from src.data.bundle import extract_data_bundle

DATA_ROOT = Path("/content/data")
EXPERIMENTS_ROOT = DRIVE_ROOT / "experiments"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["WASTE_DATA_ROOT"] = str(DATA_ROOT)
os.environ["WASTE_EXPERIMENTS_ROOT"] = str(EXPERIMENTS_ROOT)

if not BUNDLE.exists():
    raise FileNotFoundError(
        f"Chưa có bundle tại {BUNDLE}. Upload dist/waste_raw_v1.zip lên Drive trước."
    )

extract_data_bundle(BUNDLE, DATA_ROOT)
print("Data root:", DATA_ROOT)
print("Experiments:", EXPERIMENTS_ROOT)
print(sorted(p.name for p in (DATA_ROOT / "raw").iterdir()))

In [ ]:
import json

import torch

from src.utils.env import environment_info

assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))
print(json.dumps(environment_info(), ensure_ascii=False, indent=2))

In [ ]:
from src.data.download import compare_counts, count_images
from src.data.manifest import DATASETS
from src.utils.paths import dataset_dir

for name, spec in DATASETS.items():
    counts = count_images(dataset_dir(name) / spec.root_in_archive.split("/")[-1])
    problems = compare_counts(counts, spec.expected_counts)
    status = "OK" if not problems else "; ".join(problems)
    print(f"{name}: {sum(counts.values())} ảnh / {len(counts)} lớp -> {status}")

## Bước tiếp theo

Chạy training từ thư mục repo, ví dụ:

```python
!python -m research.rq1.train --config configs/rq1/<config>.yaml
```

Kết quả và checkpoint sẽ ghi trực tiếp vào `DRIVE_ROOT/experiments`.
Trước khi commit notebook, xem quy trình xuất PDF và xóa output trong `AGENTS.md`.